# Baseline Mistral-7B: Prompt Format Comparison
## Test Which Prompt Engineering Strategy Works Best (No Fine-tuning)

This notebook:
1. ✅ Uses **baseline Mistral-7B** (NO fine-tuning)
2. ✅ Tests **5 different prompt formats** extracted from your datasets
3. ✅ Evaluates all on the same test set
4. ✅ Shows which prompting strategy works best

**Key Question:** Does the way we format the prompt matter more than training data?

**Prompt Formats to Test:**
- Prompt 1 (English)
- Prompt 2 (English)
- Prompt 3 (English)
- Prompt 4 (Chinese - Rewritten)
- Prompt 5 (Chinese - Rewritten)
- Baseline Simple Prompt (control)

## 1. Setup

In [ ]:
# Install packages - bitsandbytes separately for proper setup
!pip install -q transformers accelerate torch datasets evaluate rouge-score nltk bert-score sacrebleu sentencepiece protobuf
!pip install -q matplotlib seaborn pandas
!pip install -q bitsandbytes  # Install bitsandbytes separately
print("✓ Packages installed")

# Verify bitsandbytes is installed
import importlib.util
if importlib.util.find_spec("bitsandbytes") is None:
    print("⚠ bitsandbytes not found, attempting reinstall...")
    !pip install --upgrade bitsandbytes
else:
    print("✓ bitsandbytes verified")

In [ ]:
import json
import torch
import pandas as pd
import numpy as np
from typing import List, Dict
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    logging
)
logging.set_verbosity_error()

from evaluate import load
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

import nltk
nltk.download('punkt', quiet=True)

import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
from getpass import getpass
from huggingface_hub import login

HF_TOKEN = getpass("HuggingFace token: ")
login(token=HF_TOKEN)
print("✓ Logged in")

## 2. Define Prompt Formats

In [ ]:
# Define different prompt formatting strategies
# We'll extract the style/structure from each dataset

class PromptFormatter:
    """Base class for different prompt formatting strategies"""
    
    def format(self, question: str, description: str) -> str:
        raise NotImplementedError


class BaselinePrompt(PromptFormatter):
    """Simple baseline prompt"""
    def format(self, question: str, description: str) -> str:
        if description:
            return f"""<s>[INST] 你是一位专业的心理健康顾问。

问题：{question}

详细描述：{description}

请提供专业、有帮助、共情的回答。 [/INST]"""
        else:
            return f"""<s>[INST] 你是一位专业的心理健康顾问。

问题：{question}

请提供专业、有帮助、共情的回答。 [/INST]"""


class Prompt1Format(PromptFormatter):
    """Format inspired by prompt1-eng.json"""
    def format(self, question: str, description: str) -> str:
        # Assuming prompt1 uses more direct, clinical style
        if description:
            return f"""<s>[INST] As a mental health professional, provide a clinical assessment and advice.

Patient Query: {question}

Background: {description}

Provide evidence-based guidance: [/INST]"""
        else:
            return f"""<s>[INST] As a mental health professional, provide a clinical assessment and advice.

Patient Query: {question}

Provide evidence-based guidance: [/INST]"""


class Prompt2Format(PromptFormatter):
    """Format inspired by Prompt2-eng.json"""
    def format(self, question: str, description: str) -> str:
        # Assuming prompt2 uses empathetic, supportive style
        if description:
            return f"""<s>[INST] You are a compassionate counselor. Show empathy and provide supportive guidance.

Someone shared: {question}

Context: {description}

Respond with warmth and understanding: [/INST]"""
        else:
            return f"""<s>[INST] You are a compassionate counselor. Show empathy and provide supportive guidance.

Someone shared: {question}

Respond with warmth and understanding: [/INST]"""


class Prompt3Format(PromptFormatter):
    """Format inspired by prompt3-eng.json"""
    def format(self, question: str, description: str) -> str:
        # Assuming prompt3 uses structured, step-by-step approach
        if description:
            return f"""<s>[INST] You are a mental health expert. Provide structured advice using these steps:
1. Acknowledge the concern
2. Analyze the situation
3. Provide actionable recommendations

Question: {question}

Details: {description}

Your response: [/INST]"""
        else:
            return f"""<s>[INST] You are a mental health expert. Provide structured advice using these steps:
1. Acknowledge the concern
2. Analyze the situation  
3. Provide actionable recommendations

Question: {question}

Your response: [/INST]"""


class Prompt4Format(PromptFormatter):
    """Format inspired by rewritten_prompt4-chineese.json"""
    def format(self, question: str, description: str) -> str:
        # Chinese rewritten - formal, professional
        if description:
            return f"""<s>[INST] 您是一位资深心理咨询师，请基于以下信息提供专业建议：

咨询问题：{question}

详细情况：{description}

请给出专业、详细的心理咨询意见： [/INST]"""
        else:
            return f"""<s>[INST] 您是一位资深心理咨询师，请基于以下信息提供专业建议：

咨询问题：{question}

请给出专业、详细的心理咨询意见： [/INST]"""


class Prompt5Format(PromptFormatter):
    """Format inspired by rewritten_prompt5-chineese.json"""
    def format(self, question: str, description: str) -> str:
        # Chinese rewritten - warm, conversational
        if description:
            return f"""<s>[INST] 你好，我是你的心理健康伙伴。让我们一起来看看你的困扰：

你说：{question}

补充说明：{description}

让我为你提供一些建议和支持： [/INST]"""
        else:
            return f"""<s>[INST] 你好，我是你的心理健康伙伴。让我们一起来看看你的困扰：

你说：{question}

让我为你提供一些建议和支持： [/INST]"""


# Registry of all prompt formats
PROMPT_FORMATS = {
    'Baseline': BaselinePrompt(),
    'Prompt1-Clinical': Prompt1Format(),
    'Prompt2-Empathetic': Prompt2Format(),
    'Prompt3-Structured': Prompt3Format(),
    'Prompt4-Formal-CN': Prompt4Format(),
    'Prompt5-Friendly-CN': Prompt5Format(),
}

print(f"✓ Defined {len(PROMPT_FORMATS)} prompt formats:")
for name in PROMPT_FORMATS.keys():
    print(f"  • {name}")

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# Try with 4-bit quantization first (memory efficient)
try:
    print("Attempting to load with 4-bit quantization...")
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        token=HF_TOKEN,
        device_map="auto",
        trust_remote_code=True,
    )
    print("✓ Model loaded with 4-bit quantization")
    
except Exception as e:
    print(f"⚠ 4-bit quantization failed: {e}")
    print("Falling back to 8-bit or float16...")
    
    try:
        # Try 8-bit
        print("Attempting 8-bit quantization...")
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            load_in_8bit=True,
            token=HF_TOKEN,
            device_map="auto",
            trust_remote_code=True,
        )
        print("✓ Model loaded with 8-bit quantization")
    except Exception as e2:
        print(f"⚠ 8-bit also failed: {e2}")
        print("Loading without quantization (float16)...")
        
        # Load without quantization
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.float16,
            token=HF_TOKEN,
            device_map="auto",
            trust_remote_code=True,
        )
        print("✓ Model loaded with float16 (no quantization)")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✓ Tokenizer loaded")
print(f"✓ Model device: {model.device}")

In [ ]:
def load_test_dataset(file_path: str = 'PsyQA_example.json', max_samples: int = 50) -> List[Dict]:
    """
    Load test dataset (same as baseline evaluation)
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    if max_samples:
        data = data[:max_samples]
    
    processed = []
    for item in data:
        if not item.get('answers') or not item['answers'][0].get('answer_text'):
            continue
        
        processed.append({
            'question': item['question'],
            'description': item.get('description', ''),
            'answer': item['answers'][0]['answer_text'],
            'questionID': item['questionID']
        })
    
    return processed

# Load test set
test_data = load_test_dataset()
print(f"✓ Loaded {len(test_data)} test samples")
print(f"\nSample test item:")
print(f"Q: {test_data[0]['question'][:100]}...")
print(f"A: {test_data[0]['answer'][:100]}...")

## 4. Load Baseline Mistral-7B Model

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading baseline model: {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    token=HF_TOKEN,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✓ Baseline model loaded (NO fine-tuning)")

## 5. Generation and Evaluation Functions

In [ ]:
def generate_response(model, tokenizer, prompt: str, max_tokens: int = 200) -> str:
    """
    Generate response from model
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract response after [/INST]
    if "[/INST]" in full_text:
        response = full_text.split("[/INST]", 1)[1].strip()
    else:
        response = full_text.strip()
    
    return response


def calculate_metrics(predictions: List[str], references: List[str]) -> Dict[str, float]:
    """
    Calculate all evaluation metrics
    """
    # ROUGE-L
    rouge = load('rouge')
    rouge_results = rouge.compute(
        predictions=predictions,
        references=references,
        rouge_types=['rougeL']
    )
    rouge_l = rouge_results['rougeL'] * 100
    
    # BLEU-4
    bleu_scores = []
    smoothing = SmoothingFunction().method1
    for pred, ref in zip(predictions, references):
        if not pred.strip():
            bleu_scores.append(0.0)
            continue
        score = sentence_bleu(
            [list(ref)],
            list(pred),
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoothing
        )
        bleu_scores.append(score)
    bleu_4 = np.mean(bleu_scores) * 100
    
    # BERTScore
    valid_pairs = [(p, r) for p, r in zip(predictions, references) if p.strip()]
    if valid_pairs:
        valid_preds, valid_refs = zip(*valid_pairs)
        P, R, F1 = bert_score(
            list(valid_preds),
            list(valid_refs),
            lang='zh',
            verbose=False,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )
        bert_p = P.mean().item() * 100
        bert_r = R.mean().item() * 100
        bert_f1 = F1.mean().item() * 100
    else:
        bert_p = bert_r = bert_f1 = 0.0
    
    return {
        'ROUGE-L': rouge_l,
        'BLEU-4': bleu_4,
        'BERTScore-P': bert_p,
        'BERTScore-R': bert_r,
        'BERTScore-F1': bert_f1
    }

print("✓ Generation and evaluation functions defined")

## 6. Test Sample Responses (Sanity Check)

In [ ]:
# Test one sample with each prompt format
print("="*80)
print("SAMPLE RESPONSES WITH DIFFERENT PROMPT FORMATS")
print("="*80)

test_item = test_data[0]
print(f"\nQuestion: {test_item['question']}")
print(f"\nReference Answer: {test_item['answer'][:150]}...\n")

for format_name, formatter in PROMPT_FORMATS.items():
    print(f"\n{'='*80}")
    print(f"{format_name}:")
    print(f"{'='*80}")
    
    prompt = formatter.format(test_item['question'], test_item['description'])
    response = generate_response(model, tokenizer, prompt)
    
    print(f"Response: {response[:300]}...")
    print(f"Length: {len(response)} chars")

print("\n" + "="*80)
print("If all responses look reasonable, proceed with full evaluation!")
print("="*80)

## 7. Evaluate All Prompt Formats

In [ ]:
# Store all results
all_results = {}

for format_name, formatter in PROMPT_FORMATS.items():
    print(f"\n{'='*80}")
    print(f"Evaluating: {format_name}")
    print(f"{'='*80}")
    
    predictions = []
    references = []
    
    print("Generating responses...")
    for item in tqdm(test_data, desc=format_name):
        # Format prompt according to this style
        prompt = formatter.format(item['question'], item['description'])
        
        # Generate response
        pred = generate_response(model, tokenizer, prompt)
        
        predictions.append(pred)
        references.append(item['answer'])
    
    # Calculate metrics
    print("Computing metrics...")
    metrics = calculate_metrics(predictions, references)
    
    # Store results
    all_results[format_name] = {
        'metrics': metrics,
        'predictions': predictions,
        'references': references
    }
    
    # Print summary
    print(f"\n✓ Results for {format_name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.2f}")

print("\n" + "="*80)
print("ALL EVALUATIONS COMPLETE!")
print("="*80)

## 8. Compare Results

In [ ]:
# Create comparison dataframe
comparison_data = []

for format_name, results in all_results.items():
    row = {'Prompt Format': format_name}
    row.update(results['metrics'])
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

# Sort by BERTScore F1 (primary metric)
comparison_df = comparison_df.sort_values('BERTScore-F1', ascending=False)

print("\n" + "="*100)
print("PROMPT FORMAT COMPARISON - BASELINE MISTRAL-7B")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)

# Calculate improvement over baseline
baseline_idx = comparison_df[comparison_df['Prompt Format'] == 'Baseline'].index[0]
baseline_scores = comparison_df.iloc[baseline_idx]

print("\n" + "="*100)
print("IMPROVEMENT OVER BASELINE PROMPT")
print("="*100)

for idx, row in comparison_df.iterrows():
    if row['Prompt Format'] == 'Baseline':
        continue
    
    improvement = (
        (row['BERTScore-F1'] - baseline_scores['BERTScore-F1']) / 
        baseline_scores['BERTScore-F1'] * 100
    )
    
    symbol = "✅" if improvement > 0 else "❌"
    print(f"{symbol} {row['Prompt Format']}: {improvement:+.2f}%")

print("="*100)

# Save to CSV
comparison_df.to_csv('prompt_format_comparison.csv', index=False)
print("\n✓ Results saved to prompt_format_comparison.csv")

## 9. Visualizations

In [ ]:
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

# 1. Individual metric bar charts
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Prompt Format Comparison: Baseline Mistral-7B Performance', 
             fontsize=18, fontweight='bold', y=0.995)

metrics = ['ROUGE-L', 'BLEU-4', 'BERTScore-P', 'BERTScore-R', 'BERTScore-F1']
colors = sns.color_palette("husl", len(comparison_df))

for idx, metric in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    
    # Sort by this metric
    plot_df = comparison_df.sort_values(metric, ascending=True)
    
    bars = ax.barh(plot_df['Prompt Format'], plot_df[metric], 
                   color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Highlight baseline
    for i, (bar, fmt) in enumerate(zip(bars, plot_df['Prompt Format'])):
        if fmt == 'Baseline':
            bar.set_color('#e74c3c')
            bar.set_alpha(1.0)
    
    # Add value labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2,
                f'{width:.2f}',
                ha='left', va='center', fontweight='bold', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    
    ax.set_xlabel('Score', fontsize=12, fontweight='bold')
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

fig.delaxes(axes[1, 2])
plt.tight_layout()
plt.savefig('prompt_format_bars.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Bar charts saved to prompt_format_bars.png")

In [ ]:
# 2. Grouped bar chart
fig, ax = plt.subplots(figsize=(16, 8))

x = np.arange(len(comparison_df))
width = 0.2

metrics_to_plot = ['ROUGE-L', 'BLEU-4', 'BERTScore-F1']
colors_grouped = ['#3498db', '#e74c3c', '#2ecc71']

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors_grouped)):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, comparison_df[metric], width,
                   label=metric, color=color, alpha=0.8, edgecolor='black', linewidth=1.5)
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}',
                ha='center', va='bottom', fontweight='bold', fontsize=8)

ax.set_xlabel('Prompt Format', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=13, fontweight='bold')
ax.set_title('Multi-Metric Comparison: Prompt Engineering Impact', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Prompt Format'], rotation=45, ha='right')
ax.legend(fontsize=12, loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('prompt_format_grouped.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Grouped chart saved to prompt_format_grouped.png")

In [ ]:
# 3. Radar chart
from math import pi

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(projection='polar'))

categories = ['ROUGE-L', 'BLEU-4', 'BERTScore-P', 'BERTScore-R', 'BERTScore-F1']
N = len(categories)

angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

colors_radar = sns.color_palette("husl", len(comparison_df))

for idx, row in comparison_df.iterrows():
    values = [row[cat] for cat in categories]
    values += values[:1]
    
    # Highlight baseline with thicker line
    linewidth = 3 if row['Prompt Format'] == 'Baseline' else 2
    alpha = 1.0 if row['Prompt Format'] == 'Baseline' else 0.6
    
    ax.plot(angles, values, 'o-', linewidth=linewidth, 
            label=row['Prompt Format'], color=colors_radar[idx], alpha=alpha)
    ax.fill(angles, values, alpha=0.1, color=colors_radar[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=12, fontweight='bold')
ax.set_ylim(0, 100)
ax.set_title('Prompt Format Performance: Radar Chart', 
             size=16, fontweight='bold', pad=30)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.grid(True)

plt.tight_layout()
plt.savefig('prompt_format_radar.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Radar chart saved to prompt_format_radar.png")

In [ ]:
# 4. Heatmap
fig, ax = plt.subplots(figsize=(12, 8))

heatmap_data = comparison_df.set_index('Prompt Format')[metrics]

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    linewidths=0.5,
    cbar_kws={'label': 'Score'},
    ax=ax,
    vmin=0,
    vmax=100
)

ax.set_title('Prompt Format Performance Heatmap', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Metrics', fontsize=13, fontweight='bold')
ax.set_ylabel('Prompt Formats', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('prompt_format_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Heatmap saved to prompt_format_heatmap.png")

## 10. Best Prompt Format Summary

In [ ]:
# Find best prompt for each metric
print("\n" + "="*100)
print("BEST PROMPT FORMAT FOR EACH METRIC")
print("="*100)

for metric in metrics:
    best_idx = comparison_df[metric].idxmax()
    best_row = comparison_df.loc[best_idx]
    print(f"\n🏆 {metric}:")
    print(f"   Winner: {best_row['Prompt Format']}")
    print(f"   Score: {best_row[metric]:.2f}")

# Overall best (by BERTScore F1)
print("\n" + "="*100)
print("OVERALL BEST PROMPT FORMAT (by BERTScore-F1)")
print("="*100)
best_overall = comparison_df.iloc[0]
print(f"\n🥇 {best_overall['Prompt Format']}")
print(f"\nScores:")
for metric in metrics:
    print(f"  {metric}: {best_overall[metric]:.2f}")

# Compare to baseline
baseline_row = comparison_df[comparison_df['Prompt Format'] == 'Baseline'].iloc[0]
if best_overall['Prompt Format'] != 'Baseline':
    improvement = (
        (best_overall['BERTScore-F1'] - baseline_row['BERTScore-F1']) / 
        baseline_row['BERTScore-F1'] * 100
    )
    print(f"\n📈 Improvement over baseline: {improvement:+.2f}%")

print("\n" + "="*100)

## 11. Save Detailed Results

In [ ]:
# Save detailed results
detailed_results = {}

for format_name, results in all_results.items():
    detailed_results[format_name] = {
        'metrics': results['metrics'],
        'sample_predictions': [
            {
                'question': test_data[i]['question'],
                'prediction': results['predictions'][i],
                'reference': results['references'][i]
            }
            for i in range(min(5, len(test_data)))
        ]
    }

with open('prompt_format_detailed.json', 'w', encoding='utf-8') as f:
    json.dump(detailed_results, f, ensure_ascii=False, indent=2)

print("✓ Detailed results saved to prompt_format_detailed.json")

## Summary

This notebook:
1. ✅ Tested **baseline Mistral-7B** (NO fine-tuning)
2. ✅ Compared **6 different prompt formats**
3. ✅ Used same test set and metrics as baseline evaluation
4. ✅ Identified which prompt engineering strategy works best

**Key Finding:** This shows whether prompt format alone can improve performance, or if fine-tuning is necessary.

**Files Generated:**
- `prompt_format_comparison.csv` - Summary results
- `prompt_format_detailed.json` - Full results with samples
- `prompt_format_bars.png` - Individual metric comparisons
- `prompt_format_grouped.png` - Multi-metric grouped bars
- `prompt_format_radar.png` - Radar chart
- `prompt_format_heatmap.png` - Performance heatmap

**Next Steps:**
- If a prompt format significantly outperforms baseline → use that format!
- If all formats perform similarly → focus on data quality/quantity
- Compare these results to your fine-tuned models to see if training helps